In [1]:
import os
import nibabel as nib
import json
from pathlib import Path
import shutil

import glob
import argparse
import multiprocessing
import shutil
from typing import Optional
import SimpleITK as sitk
from batchgenerators.utilities.file_and_folder_operations import *
from nnunetv2.paths import nnUNet_raw
from nnunetv2.utilities.dataset_name_id_conversion import find_candidate_datasets
from nnunetv2.configuration import default_num_processes
import numpy as np
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json


In [2]:

# extracted traiing.zip file is here
base = '/home/qingyu/datasets/ds004199-1.0.6'
target_dataset_id = 106
target_dataset_name = f'Dataset{target_dataset_id:03.0f}_FCD'
participants = join(base, 'participants.tsv')


maybe_mkdir_p(join(nnUNet_raw, target_dataset_name))
imagesTr = join(nnUNet_raw, target_dataset_name, 'imagesTr')
imagesTs = join(nnUNet_raw, target_dataset_name, 'imagesTs')
labelsTr = join(nnUNet_raw, target_dataset_name, 'labelsTr')
maybe_mkdir_p(imagesTr)
maybe_mkdir_p(imagesTs)
maybe_mkdir_p(labelsTr)

In [94]:
import pandas as pd
 

def read_csv(tsv_file: str):
    df = pd.read_csv(participants, delimiter='\t', header=0)
    # df=df.dropna(axis=0, how='any',subset=['lobe'])
    train_rows = df[df['split'] == 'train']['participant_id'].values.tolist()
    test_rows = df[df['split'] == 'test']['participant_id'].values.tolist()
    return train_rows, test_rows

train_rows, test_rows = read_csv(participants)

In [ ]:
def register(sub,is_tr=False)
    t1w_files = layout.get(subject=sub, suffix="T1w", extension=[".nii", ".nii.gz"], return_type='file')
    flair_files = layout.get(subject=sub, suffix="FLAIR", extension=[".nii", ".nii.gz"], return_type='file')

    if not t1w_files or not flair_files:
        print(f"Skipping {sub} — missing T1w or FLAIR.")
        continue

    t1_file = t1w_files[0]
    flair_file = flair_files[0]

    # Handle non-BIDS ROI manually (e.g., sub-001/anat/sub-001_FLAIR_roi.nii.gz)
    roi_candidates = glob.glob(join(base, f"sub-{sub}", 'anat', '*FLAIR_roi.nii.gz'))
    roi_file = roi_candidates[0] if roi_candidates else None

    # Read and register images
    t1_img = ants.image_read(str(t1_file))
    flair_img = ants.image_read(str(flair_file))

    t1_img_nib = nib.load(str(t1_file))

    
    tx = ants.registration(fixed=t1_img, moving=flair_img, type_of_transform="Affine")
    flair_reg = tx["warpedmovout"]

    # Save nnU-Net modalities
    out_base = f"sub-{sub}"
    # nib.save(nib.Nifti1Image(t1_img.numpy(), t1_img.affine), imagesTr / f"{out_base}_0000.nii.gz")
    # nib.save(nib.Nifti1Image(flair_reg.numpy(), t1_img.affine), imagesTr / f"{out_base}_0001.nii.gz")
    nib.save(nib.Nifti1Image(t1_img.numpy(), t1_img_nib.affine), join(imagesTr , f"{out_base}_0000.nii.gz"))
    nib.save(nib.Nifti1Image(flair_reg.numpy(), t1_img_nib.affine), join(imagesTr , f"{out_base}_0001.nii.gz"))

    
    # Process ROI
    if roi_file:
        roi_img = ants.image_read(str(roi_file))
        roi_reg = ants.apply_transforms(fixed=t1_img, moving=roi_img,
                                        transformlist=tx['fwdtransforms'],
                                        interpolator='nearestNeighbor')
        roi_data = roi_reg.numpy().astype(np.uint8)
    else:
        roi_data = np.zeros(t1_img.shape, dtype=np.uint8)

    nib.save(nib.Nifti1Image(roi_data, t1_img_nib.affine), join(labelsTr , f"{out_base}.nii.gz"))






In [97]:
cases = subdirs(base, join=False)
i = 0
for case in cases:
    if case in train_rows:
        shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
        shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTr, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
        shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR_roi.nii.gz'))[0], join(labelsTr, 'FCD_' + case.split('-')[1] + '.nii.gz'))
    elif case in test_rows:   
        shutil.copy(glob.glob(join(base, case, 'anat', '*T1w.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0000.nii.gz'))
        shutil.copy(glob.glob(join(base, case, 'anat', '*FLAIR.nii.gz'))[0], join(imagesTs, 'FCD_' + case.split('-')[1] + '_0001.nii.gz'))
    else:
        i+=1
        # print(case)
        # print('error')

    
        
    



In [101]:
out_dir = join(nnUNet_raw, target_dataset_name)
generate_dataset_json(
    out_dir,
    channel_names={
        0: "cineMRI",

         0: "T1",
        1: "FLAIR"
    },
    labels={
        "background": 0,
        "FCD": 1
    },
    file_ending=".nii.gz",
    num_training_cases=len(train_rows),
)